In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

karthikaproject_flyrank_ml05_training_path = kagglehub.dataset_download('karthikaproject/flyrank-ml05-training')
karthikaproject_flyrank_ml05_test_june2026_path = kagglehub.dataset_download('karthikaproject/flyrank-ml05-test-june2026')
karthikaproject_w03_feature_leakage_check_ipynb_path = kagglehub.notebook_output_download('karthikaproject/w03-feature-leakage-check-ipynb')

print('Data source import complete.')


# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")

import os
os.environ["HF_TOKEN"] = secret_value_0

import duckdb
import pandas as pd
import numpy as np
import time
import json
import subprocess
import shutil

from huggingface_hub import whoami

try:
    identity = whoami(token=os.environ["HF_TOKEN"])
    print("HF token is valid. Authenticated as:", identity.get("name", identity))
except Exception as e:
    print("HF token FAILED validation:", e)
    print("Check: Add-ons -> Secrets -> HF_TOKEN is added AND toggled ON for this notebook.")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
daily_path = f"{rel}/fact_content_daily_performance/month=*/data_0.parquet"

print("Setup complete. con, rel, and daily_path are ready.")

HF token is valid. Authenticated as: Karthikaakkappilly
Setup complete. con, rel, and daily_path are ready.


In [ ]:
# Data diagnostics

schema = con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{daily_path}')
""").df()

print("FULL SCHEMA")
print(schema)

unique_clients = con.sql(f"""
    SELECT COUNT(DISTINCT client_hash_id) AS unique_clients
    FROM read_parquet('{daily_path}')
""").df()

print("\nUnique clients")
print(unique_clients)

unique_pages = con.sql(f"""
    SELECT COUNT(DISTINCT (client_hash_id, content_hash_id)) AS unique_client_page_pairs
    FROM read_parquet('{daily_path}')
""").df()

print("\nUnique (client, page) pairs")
print(unique_pages)

FULL SCHEMA
                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  Non

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Unique clients
   unique_clients
0              70


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Unique (client, page) pairs
   unique_client_page_pairs
0                    427292


In [ ]:
# GSC position value diagnostics

position_diagnostics = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_avg_position IS NULL THEN 'NaN'
            WHEN gsc_avg_position = 0 THEN '0'
            WHEN gsc_avg_position > 0 THEN '> 0'
            ELSE '< 0'
        END AS position_status,

        CASE
            WHEN gsc_impressions IS NULL THEN 'NaN'
            WHEN gsc_impressions = 0 THEN '0'
            WHEN gsc_impressions > 0 THEN '> 0'
            ELSE '< 0'
        END AS impressions_status,

        COUNT(*) AS row_count

    FROM read_parquet('{daily_path}')

    GROUP BY
        position_status,
        impressions_status

    ORDER BY
        position_status,
        impressions_status
""").df()

print("GSC POSITION × IMPRESSIONS DIAGNOSTICS")
display(position_diagnostics)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

GSC POSITION × IMPRESSIONS DIAGNOSTICS


,position_status,impressions_status,row_count
0,0,> 0,1393857
1,> 0,> 0,27576144
2,NaN,0,49767598
3,NaN,> 0,50
4,NaN,NaN,98006


In [ ]:
# Training window creation

HISTORY_SIZE = 60
TARGET_SIZE = 30
BLOCK_SIZE = HISTORY_SIZE + TARGET_SIZE
STRIDE = 15

MODEL_COLS = [
    "gsc_impressions", "gsc_clicks", "gsc_avg_position",
    "ga4_pageviews", "ga4_sessions", "ga4_users",
    "ga4_engaged_sessions", "ga4_total_engagement_sec",
    "sessions_organic", "sessions_direct", "sessions_referral",
    "sessions_social", "sessions_paid", "sessions_ai",
    "scroll_events",
]

select_cols = ",\n            ".join(
    ["client_hash_id", "content_hash_id", "report_date"]
    + [
        "gsc_impressions", "gsc_clicks", "gsc_sum_position", "gsc_avg_position",
        "ga4_pageviews", "ga4_sessions", "ga4_users",
        "ga4_engaged_sessions", "ga4_total_engagement_sec",
        "sessions_organic", "sessions_direct", "sessions_referral",
        "sessions_social", "sessions_paid", "sessions_ai",
        "scroll_events",
    ]
)

sum_cols = [
    "gsc_impressions", "gsc_clicks",
    "ga4_pageviews", "ga4_sessions", "ga4_users",
    "ga4_engaged_sessions", "ga4_total_engagement_sec",
    "sessions_organic", "sessions_direct", "sessions_referral",
    "sessions_social", "sessions_paid", "sessions_ai",
    "scroll_events",
]

history_sum_aggs = ",\n            ".join(
    [f"SUM({c}) AS {c}" for c in sum_cols]
)

client_list = con.sql(f"""
    SELECT DISTINCT client_hash_id
    FROM read_parquet('{daily_path}')
    WHERE report_date < '2026-06-01'
""").df()["client_hash_id"].tolist()

print(f"Processing {len(client_list)} clients one at a time...")


def run_with_retry(query_fn, max_retries=6, base_delay=15):
    for attempt in range(max_retries):
        try:
            return query_fn()
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"    FAILED after {max_retries} attempts: {e}")
                raise

            wait = base_delay * (2 ** attempt)
            print(
                f"    Error on attempt {attempt + 1} ({e}). "
                f"Retrying in {wait}s..."
            )
            time.sleep(wait)


con.execute("SET threads=2")

LOCAL_DAILY_PATH = "/kaggle/working/daily_performance_local.parquet"

print("Downloading remote data to local disk once...")


def download_fn():
    return con.sql(f"""
        COPY (
            SELECT * FROM read_parquet('{daily_path}')
            WHERE report_date < '2026-06-01'
        ) TO '{LOCAL_DAILY_PATH}' (FORMAT PARQUET)
    """)


run_with_retry(download_fn)

size_mb = os.path.getsize(LOCAL_DAILY_PATH) / (1024 * 1024)

print(
    f"Local copy saved: "
    f"{LOCAL_DAILY_PATH} ({size_mb:.1f} MB)"
)

daily_path_local = LOCAL_DAILY_PATH

SAVE_PATH_PARQUET = "/kaggle/working/ml05_windows.parquet"

first_batch = True
total_windows = 0

for i, client_id in enumerate(client_list):

    def query_fn():
        return con.sql(f"""
        WITH ordered AS (
            SELECT
                {select_cols}
            FROM read_parquet('{daily_path_local}')
            WHERE client_hash_id = '{client_id}'
        ),

        numbered AS (
            SELECT
                *,
                ROW_NUMBER() OVER (
                    PARTITION BY client_hash_id, content_hash_id
                    ORDER BY report_date
                ) AS rn,

                COUNT(*) OVER (
                    PARTITION BY client_hash_id, content_hash_id
                ) AS max_rn

            FROM ordered
        ),

        starts AS (
            SELECT DISTINCT
                client_hash_id,
                content_hash_id,

                UNNEST(
                    GENERATE_SERIES(
                        1,
                        max_rn - {BLOCK_SIZE} + 1,
                        {STRIDE}
                    )
                ) AS window_start

            FROM numbered
            WHERE max_rn >= {BLOCK_SIZE}
        ),

        windowed AS (
            SELECT
                s.client_hash_id,
                s.content_hash_id,
                s.window_start,

                n.rn - s.window_start AS pos_in_window,

                n.report_date,

                {", ".join(["n." + c for c in [
                    "gsc_impressions", "gsc_clicks", "gsc_sum_position",
                    "gsc_avg_position",
                    "ga4_pageviews", "ga4_sessions", "ga4_users",
                    "ga4_engaged_sessions", "ga4_total_engagement_sec",
                    "sessions_organic", "sessions_direct",
                    "sessions_referral", "sessions_social",
                    "sessions_paid", "sessions_ai",
                    "scroll_events"
                ]])}

            FROM starts s

            JOIN numbered n
              ON n.client_hash_id = s.client_hash_id
             AND n.content_hash_id = s.content_hash_id
             AND n.rn BETWEEN s.window_start
                           AND s.window_start + {BLOCK_SIZE} - 1
        ),

        history AS (
            SELECT
                client_hash_id,
                content_hash_id,
                window_start,

                COUNT(*) AS history_rows,

                {history_sum_aggs},

                CASE
                    WHEN SUM(
                        CASE
                            WHEN gsc_avg_position IS NOT NULL
                            THEN gsc_impressions
                            ELSE 0
                        END
                    ) > 0
                    THEN
                        CAST(
                            SUM(
                                CASE
                                    WHEN gsc_avg_position IS NOT NULL
                                    THEN gsc_sum_position
                                    ELSE 0
                                END
                            ) AS DOUBLE
                        )
                        /
                        CAST(
                            SUM(
                                CASE
                                    WHEN gsc_avg_position IS NOT NULL
                                    THEN gsc_impressions
                                    ELSE 0
                                END
                            ) AS DOUBLE
                        )
                    ELSE NULL
                END AS gsc_avg_position,

                SUM(
                    CASE
                        WHEN gsc_impressions = 0
                         AND gsc_clicks = 0
                        THEN 1
                        ELSE 0
                    END
                ) AS zero_activity_rows,

                MIN(report_date) AS history_start,
                MAX(report_date) AS history_end

            FROM windowed
            WHERE pos_in_window < {HISTORY_SIZE}
            GROUP BY client_hash_id, content_hash_id, window_start
        ),

        target AS (
            SELECT
                client_hash_id,
                content_hash_id,
                window_start,

                COUNT(*) AS target_rows,

                CASE
                    WHEN SUM(
                        CASE
                            WHEN gsc_avg_position IS NOT NULL
                            THEN gsc_impressions
                            ELSE 0
                        END
                    ) > 0
                    THEN
                        CAST(
                            SUM(
                                CASE
                                    WHEN gsc_avg_position IS NOT NULL
                                    THEN gsc_sum_position
                                    ELSE 0
                                END
                            ) AS DOUBLE
                        )
                        /
                        CAST(
                            SUM(
                                CASE
                                    WHEN gsc_avg_position IS NOT NULL
                                    THEN gsc_impressions
                                    ELSE 0
                                END
                            ) AS DOUBLE
                        )
                    ELSE NULL
                END AS future_avg_position,

                MIN(report_date) AS target_start,
                MAX(report_date) AS target_end

            FROM windowed
            WHERE pos_in_window >= {HISTORY_SIZE}
            GROUP BY client_hash_id, content_hash_id, window_start
        )

        SELECT
            h.*,
            t.target_rows,
            t.future_avg_position,
            t.target_start,
            t.target_end

        FROM history h

        LEFT JOIN target t
          ON h.client_hash_id = t.client_hash_id
         AND h.content_hash_id = t.content_hash_id
         AND h.window_start = t.window_start

        WHERE h.history_rows = {HISTORY_SIZE}

        """).df()

    batch = run_with_retry(query_fn)

    if len(batch) > 0:

        if first_batch:
            batch.to_parquet(
                SAVE_PATH_PARQUET,
                index=False
            )
            first_batch = False

        else:
            existing = pd.read_parquet(SAVE_PATH_PARQUET)

            combined = pd.concat(
                [existing, batch],
                ignore_index=True
            )

            combined.to_parquet(
                SAVE_PATH_PARQUET,
                index=False
            )

            del existing, combined

        total_windows += len(batch)

    del batch

    if (i + 1) % 10 == 0 or (i + 1) == len(client_list):
        print(
            f"  Processed {i + 1}/{len(client_list)} clients "
            f"— total windows so far: {total_windows}"
        )

print(f"\nDone. Total windows built: {total_windows}")

windows = pd.read_parquet(SAVE_PATH_PARQUET)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Processing 70 clients one at a time...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Local copy saved: /kaggle/working/daily_performance_local.parquet (1392.6 MB)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Processed 70/70 clients — total windows so far: 2408133

Done. Total windows built: 2408133


In [ ]:
# Save ML-05 training dataset to Kaggle

source_file = "/kaggle/working/ml05_windows.parquet"

dataset_dir = "/kaggle/working/flyrank-ml05-training"
os.makedirs(dataset_dir, exist_ok=True)

shutil.copy2(
    source_file,
    os.path.join(dataset_dir, "ml05_windows.parquet")
)

metadata = {
    "title": "FlyRank ML05 Training",
    "id": "karthikaproject/flyrank-ml05-training",
    "licenses": [
        {
            "name": "CC0-1.0"
        }
    ],
    "subtitle": "ML-05 historical feature and future target windows",
    "description": "Generated ML-05 training windows for the FlyRank ML internship project.",
    "isPrivate": True
}

with open(
    os.path.join(dataset_dir, "dataset-metadata.json"),
    "w"
) as f:
    json.dump(metadata, f, indent=2)

result = subprocess.run(
    ["kaggle", "datasets", "create", "-p", dataset_dir],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

Starting upload for file ml05_windows.parquet
Upload successful: ml05_windows.parquet (53MB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/karthikaproject/flyrank-ml05-training


  0%|          | 0.00/52.8M [00:00<?, ?B/s]
  2%|▏         | 1.03M/52.8M [00:00<00:05, 9.87MB/s]
 14%|█▍        | 7.56M/52.8M [00:00<00:01, 41.5MB/s]
 52%|█████▏    | 27.4M/52.8M [00:00<00:00, 115MB/s] 
 73%|███████▎  | 38.6M/52.8M [00:00<00:00, 83.6MB/s]
 90%|█████████ | 47.7M/52.8M [00:00<00:00, 84.8MB/s]
100%|██████████| 52.8M/52.8M [00:01<00:00, 41.9MB/s]



In [ ]:
# Access and validate saved ML-05 training dataset

FILE_PATH = "/kaggle/input/datasets/karthikaproject/flyrank-ml05-training/ml05_windows.parquet"

try:
    df = pd.read_parquet(FILE_PATH)
    print("File opened successfully. No corruption.")
except Exception as e:
    print("FILE FAILED TO OPEN:", e)
    raise

print("\nShape (rows, columns):", df.shape)

print("\nColumn list:")
print(df.columns.tolist())

print("\nFirst 3 rows:")
print(df.head(3))

print("\nQuick sanity check on key columns:")

for col in [
    "client_hash_id",
    "content_hash_id",
    "history_rows",
    "gsc_avg_position",
    "future_avg_position"
]:
    if col in df.columns:
        print(
            f"  {col}: "
            f"{df[col].notna().sum()} non-null out of {len(df)}"
        )
    else:
        print(f"  {col}: COLUMN MISSING")

if "history_rows" in df.columns:
    bad_rows = (df["history_rows"] != 60).sum()
    print(
        f"\nRows where history_rows != 60: "
        f"{bad_rows} (should be 0)"
    )

File opened successfully. No corruption.

Shape (rows, columns): (2408133, 26)

Column list:
['client_hash_id', 'content_hash_id', 'window_start', 'history_rows', 'gsc_impressions', 'gsc_clicks', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'scroll_events', 'gsc_avg_position', 'zero_activity_rows', 'history_start', 'history_end', 'target_rows', 'future_avg_position', 'target_start', 'target_end']

First 3 rows:
            client_hash_id           content_hash_id  window_start  \
0  client_ff644d8251367cbb  content_00bb2e1e87247dfd            91   
1  client_ff644d8251367cbb  content_052cb1e224fce649           151   
2  client_ff644d8251367cbb  content_08dc7073bf2cdeb5            91   

   history_rows  gsc_impressions  gsc_clicks  ga4_pageviews  ga4_sessions  \
0            60              2.0         0.0            0.0    

In [ ]:
# Validate position coverage

total = len(df)

history_valid = df["gsc_avg_position"].notna().sum()
target_valid = df["future_avg_position"].notna().sum()

both_valid = (
    df["gsc_avg_position"].notna()
    & df["future_avg_position"].notna()
).sum()

neither_valid = (
    df["gsc_avg_position"].isna()
    & df["future_avg_position"].isna()
).sum()

only_history = (
    df["gsc_avg_position"].notna()
    & df["future_avg_position"].isna()
).sum()

only_target = (
    df["gsc_avg_position"].isna()
    & df["future_avg_position"].notna()
).sum()

print("Total windows:", total)
print(
    f"History valid (gsc_avg_position not null): "
    f"{history_valid} ({history_valid/total*100:.1f}%)"
)
print(
    f"Target valid (future_avg_position not null): "
    f"{target_valid} ({target_valid/total*100:.1f}%)"
)
print(
    f"\nBOTH valid (usable for training): "
    f"{both_valid} ({both_valid/total*100:.1f}%)"
)
print(
    f"Only history valid, target missing: "
    f"{only_history} ({only_history/total*100:.1f}%)"
)
print(
    f"Only target valid, history missing: "
    f"{only_target} ({only_target/total*100:.1f}%)"
)
print(
    f"Neither valid (fully unusable): "
    f"{neither_valid} ({neither_valid/total*100:.1f}%)"
)

Total windows: 2408133
History valid (gsc_avg_position not null): 1411120 (58.6%)
Target valid (future_avg_position not null): 1337969 (55.6%)

BOTH valid (usable for training): 1247092 (51.8%)
Only history valid, target missing: 164028 (6.8%)
Only target valid, history missing: 90877 (3.8%)
Neither valid (fully unusable): 906136 (37.6%)


In [ ]:
# Sealed June 2026 test set

TEST_TARGET_START = "2026-06-01"
TEST_TARGET_END = "2026-06-30"

test_windows = con.sql(f"""
    WITH ordered AS (
        SELECT {select_cols}
        FROM read_parquet('{daily_path}')
    ),

    history_ranked AS (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY client_hash_id, content_hash_id
                ORDER BY report_date DESC
            ) AS rn_desc

        FROM ordered

        WHERE report_date < '{TEST_TARGET_START}'
    ),

    history AS (
        SELECT
            client_hash_id,
            content_hash_id,
            COUNT(*) AS history_rows,

            {history_sum_aggs},

            CASE
                WHEN SUM(
                    CASE
                        WHEN gsc_avg_position IS NOT NULL
                        THEN gsc_impressions
                        ELSE 0
                    END
                ) > 0
                THEN
                    CAST(
                        SUM(
                            CASE
                                WHEN gsc_avg_position IS NOT NULL
                                THEN gsc_sum_position
                                ELSE 0
                            END
                        ) AS DOUBLE
                    )
                    /
                    CAST(
                        SUM(
                            CASE
                                WHEN gsc_avg_position IS NOT NULL
                                THEN gsc_impressions
                                ELSE 0
                            END
                        ) AS DOUBLE
                    )
                ELSE NULL
            END AS gsc_avg_position,

            SUM(
                CASE
                    WHEN gsc_impressions = 0
                     AND gsc_clicks = 0
                    THEN 1
                    ELSE 0
                END
            ) AS zero_activity_rows,

            MIN(report_date) AS history_start,
            MAX(report_date) AS history_end

        FROM history_ranked

        WHERE rn_desc <= {HISTORY_SIZE}

        GROUP BY
            client_hash_id,
            content_hash_id

        HAVING COUNT(*) = {HISTORY_SIZE}
    ),

    target AS (
        SELECT
            client_hash_id,
            content_hash_id,
            COUNT(*) AS target_rows,

            CASE
                WHEN SUM(
                    CASE
                        WHEN gsc_avg_position IS NOT NULL
                        THEN gsc_impressions
                        ELSE 0
                    END
                ) > 0
                THEN
                    CAST(
                        SUM(
                            CASE
                                WHEN gsc_avg_position IS NOT NULL
                                THEN gsc_sum_position
                                ELSE 0
                            END
                        ) AS DOUBLE
                    )
                    /
                    CAST(
                        SUM(
                            CASE
                                WHEN gsc_avg_position IS NOT NULL
                                THEN gsc_impressions
                                ELSE 0
                            END
                        ) AS DOUBLE
                    )
                ELSE NULL
            END AS future_avg_position,

            MIN(report_date) AS target_start,
            MAX(report_date) AS target_end

        FROM ordered

        WHERE report_date BETWEEN
            '{TEST_TARGET_START}'
            AND '{TEST_TARGET_END}'

        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT
        h.*,
        t.target_rows,
        t.future_avg_position,
        t.target_start,
        t.target_end

    FROM history h

    LEFT JOIN target t
        ON h.client_hash_id = t.client_hash_id
       AND h.content_hash_id = t.content_hash_id

""").df()

before_filter = len(test_windows)

test_windows = test_windows[
    test_windows["target_rows"] == 30
].copy()

after_filter = len(test_windows)

TEST_SAVE_PATH = "/kaggle/working/ml05_test_june2026.parquet"

test_windows.to_parquet(
    TEST_SAVE_PATH,
    index=False
)

print("\nSEALED TEST SET (JUNE 2026)")
print(f"Test rows before full-June filter: {before_filter}")
print(f"Test rows after filter (target_rows == 30): {after_filter}")
print(f"Saved test set to: {TEST_SAVE_PATH}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


SEALED TEST SET (JUNE 2026)
Test rows before full-June filter: 350237
Test rows after filter (target_rows == 30): 271333
Saved test set to: /kaggle/working/ml05_test_june2026.parquet


In [ ]:
# Save ML-05 June test dataset to Kaggle

source_file = "/kaggle/working/ml05_test_june2026.parquet"

dataset_dir = "/kaggle/working/flyrank-ml05-test-june2026"
os.makedirs(dataset_dir, exist_ok=True)

shutil.copy2(
    source_file,
    os.path.join(dataset_dir, "ml05_test_june2026.parquet")
)

metadata = {
    "title": "FlyRank ML05 June 2026 Test",
    "id": "karthikaproject/flyrank-ml05-test-june2026",
    "licenses": [
        {
            "name": "CC0-1.0"
        }
    ],
    "subtitle": "ML-05 sealed June 2026 test dataset",
    "description": "Sealed June 2026 test dataset with 60 historical records before June and June 2026 as the target period.",
    "isPrivate": True
}

with open(
    os.path.join(dataset_dir, "dataset-metadata.json"),
    "w"
) as f:
    json.dump(metadata, f, indent=2)

result = subprocess.run(
    ["kaggle", "datasets", "create", "-p", dataset_dir],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

Starting upload for file ml05_test_june2026.parquet
Upload successful: ml05_test_june2026.parquet (9MB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/karthikaproject/flyrank-ml05-test-june2026


  0%|          | 0.00/8.65M [00:00<?, ?B/s]
 14%|█▍        | 1.22M/8.65M [00:00<00:00, 12.6MB/s]
100%|██████████| 8.65M/8.65M [00:00<00:00, 11.1MB/s]



In [ ]:
# Validate ML-05 June test dataset

FILE_PATH = "/kaggle/input/datasets/karthikaproject/flyrank-ml05-test-june2026/ml05_test_june2026.parquet"

test_check = pd.read_parquet(FILE_PATH)

print("Test dataset shape:", test_check.shape)

print("\nTarget rows distribution:")
print(test_check["target_rows"].value_counts().sort_index())

print("\nHistory rows distribution:")
print(test_check["history_rows"].value_counts().sort_index())

print("\nTarget start:")
print(test_check["target_start"].min())

print("\nTarget end:")
print(test_check["target_end"].max())

Test dataset shape: (271333, 25)

Target rows distribution:
target_rows
30    271333
Name: count, dtype: Int64

History rows distribution:
history_rows
60    271333
Name: count, dtype: int64

Target start:
2026-06-01 00:00:00

Target end:
2026-06-30 00:00:00


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
# Feature notes

feature_notes = [
    ["gsc_impressions", "Google Search Console impressions for the page", "Missing/zero retained as observed", "No", "Yes — historical window"],
    ["gsc_clicks", "Google Search Console clicks for the page", "Missing/zero retained as observed", "No", "Yes — historical window"],
    ["gsc_avg_position", "Impression-weighted average Google Search position", "Missing position retained; ignored only in position aggregation", "No", "Yes — historical window"],
    ["ga4_pageviews", "Number of page views recorded by GA4", "Missing/zero retained as observed", "No", "Yes — historical window"],
    ["ga4_sessions", "Total GA4 sessions for the page", "Missing/zero retained as observed", "No", "Yes — historical window"],
    ["ga4_users", "Number of users associated with the page", "Missing/zero retained as observed", "No", "Yes — historical window"],
    ["ga4_engaged_sessions", "Number of engaged GA4 sessions", "Missing/zero retained as observed", "No", "Yes — historical window"],
    ["ga4_total_engagement_sec", "Total engagement time in seconds", "Missing/zero retained as observed", "No", "Yes — historical window"],
    ["sessions_organic", "Sessions from organic search", "Missing/zero retained as observed", "No", "Yes — historical window"],
    ["sessions_direct", "Sessions classified as direct traffic", "Missing/zero retained as observed", "No", "Yes — historical window"],
    ["sessions_referral", "Sessions from referral traffic", "Missing/zero retained as observed", "No", "Yes — historical window"],
    ["sessions_social", "Sessions from social traffic", "Missing/zero retained as observed", "No", "Yes — historical window"],
    ["sessions_paid", "Sessions from paid traffic", "Missing/zero retained as observed", "No", "Yes — historical window"],
    ["sessions_ai", "Sessions attributed to AI sources", "Missing/zero retained as observed", "No", "Yes — historical window"],
    ["scroll_events", "Recorded scroll activity for the page", "Missing/zero retained as observed", "No", "Yes — historical window"]
]

feature_notes_df = pd.DataFrame(
    feature_notes,
    columns=[
        "Feature",
        "Meaning",
        "Missing-value handling",
        "Categorical",
        "Available before prediction?"
    ]
)

display(feature_notes_df)

,Feature,Meaning,Missing-value handling,Categorical,Available before prediction?
0,gsc_impressions,Google Search Console impressions for the page,Missing/zero retained as observed,No,Yes — historical window
1,gsc_clicks,Google Search Console clicks for the page,Missing/zero retained as observed,No,Yes — historical window
2,gsc_avg_position,Impression-weighted average Google Search posi...,Missing position retained; ignored only in pos...,No,Yes — historical window
3,ga4_pageviews,Number of page views recorded by GA4,Missing/zero retained as observed,No,Yes — historical window
4,ga4_sessions,Total GA4 sessions for the page,Missing/zero retained as observed,No,Yes — historical window
5,ga4_users,Number of users associated with the page,Missing/zero retained as observed,No,Yes — historical window
6,ga4_engaged_sessions,Number of engaged GA4 sessions,Missing/zero retained as observed,No,Yes — historical window
7,ga4_total_engagement_sec,Total engagement time in seconds,Missing/zero retained as observed,No,Yes — historical window
8,sessions_organic,Sessions from organic search,Missing/zero retained as observed,No,Yes — historical window
9,sessions_direct,Sessions classified as direct traffic,Missing/zero retained as observed,No,Yes — historical window


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
MODEL_COLS = [
    "gsc_impressions", "gsc_clicks", "gsc_avg_position",
    "ga4_pageviews", "ga4_sessions", "ga4_users",
    "ga4_engaged_sessions", "ga4_total_engagement_sec",
    "sessions_organic", "sessions_direct", "sessions_referral",
    "sessions_social", "sessions_paid", "sessions_ai",
    "scroll_events",
]

In [ ]:
# Leakage hunt

target_or_future_cols = [
    "future_avg_position",
    "target_rows",
    "target_start",
    "target_end"
]

print("MODEL FEATURES")
print(MODEL_COLS)

print("\nTARGET / FUTURE COLUMNS NOT USED AS FEATURES")
for col in target_or_future_cols:
    print(f"{col}: {'NOT USED' if col not in MODEL_COLS else 'LEAKAGE'}")

print("\nLEAKAGE CHECK")
leakage_cols = [col for col in target_or_future_cols if col in MODEL_COLS]

if len(leakage_cols) == 0:
    print("PASS: No target or future columns are included in MODEL_COLS.")
else:
    print("FAIL: Leakage detected:", leakage_cols)

MODEL FEATURES
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'scroll_events']

TARGET / FUTURE COLUMNS NOT USED AS FEATURES
future_avg_position: NOT USED
target_rows: NOT USED
target_start: NOT USED
target_end: NOT USED

LEAKAGE CHECK
PASS: No target or future columns are included in MODEL_COLS.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# Excluded fields and reasons

excluded_fields = [
    ["client_hash_id", "Identifier only; not a predictive feature"],
    ["content_hash_id", "Identifier only; not a predictive feature"],
    ["window_start", "Window bookkeeping; not a predictive feature"],
    ["history_rows", "Validation/bookkeeping field; not a predictive feature"],
    ["zero_activity_rows", "Derived summary/count rather than a raw model feature"],
    ["history_start", "Historical window boundary; bookkeeping only"],
    ["history_end", "Historical window boundary; bookkeeping only"],
    ["target_rows", "Target-period bookkeeping; belongs to the future window"],
    ["future_avg_position", "Target variable; using it as a feature would cause target leakage"],
    ["target_start", "Future target-period boundary; unavailable at prediction time"],
    ["target_end", "Future target-period boundary; unavailable at prediction time"]
]

excluded_fields_df = pd.DataFrame(
    excluded_fields,
    columns=["Field", "Reason for exclusion"]
)

display(excluded_fields_df)

,Field,Reason for exclusion
0,client_hash_id,Identifier only; not a predictive feature
1,content_hash_id,Identifier only; not a predictive feature
2,window_start,Window bookkeeping; not a predictive feature
3,history_rows,Validation/bookkeeping field; not a predictive...
4,zero_activity_rows,Derived summary/count rather than a raw model ...
5,history_start,Historical window boundary; bookkeeping only
6,history_end,Historical window boundary; bookkeeping only
7,target_rows,Target-period bookkeeping; belongs to the futu...
8,future_avg_position,Target variable; using it as a feature would c...
9,target_start,Future target-period boundary; unavailable at ...


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.